# Banking feedback review benchmark

A reproducible comparison using the completed review file. Run after attaching the processed data, review decisions and research source as private Kaggle inputs.


In [ ]:
import json
import sys
import time
from pathlib import Path
started=time.monotonic()
source=next(Path('/kaggle/input').rglob('run_experiments.py')).parent
sys.path.insert(0,str(source))
print('RESOLVED_SOURCE',str(source))


## Source and runtime checks


In [ ]:
import torch
from source_audit import source_audit
for path in source.glob('*.py'):
    print('SOURCE_AUDIT',source_audit(path))
print('RUNTIME',json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))


## Review data and fixed protocol


In [ ]:
data=next(Path('/kaggle/input').rglob('labelled_reviews.parquet'))
reviews=next(Path('/kaggle/input').rglob('review_decisions.json'))
print('RESOLVED_DATA',str(data),str(reviews))
from protocol import prepare_gold
config=json.loads((source/'experiment_config.json').read_text())
gold,manifest,split_hash=prepare_gold(data,reviews,config)
print('SPLIT_HASH',split_hash)
print(gold.groupby(['split','reviewer_sentiment']).size().unstack(fill_value=0))


## Training, validation comparison and locked test

The runner displays per-model reports and figures from common validation rows, locks the selected model and threshold, verifies reload, and evaluates the reserved test set once.


In [ ]:
from run_experiments import execute
try:
    result=execute(data,reviews,'/kaggle/working/banklens_experiments')
finally:
    print('TOTAL_NOTEBOOK_SECONDS',round(time.monotonic()-started,3))


## Retained export


In [ ]:
print(json.dumps(result,indent=2))
print('ARTIFACT_PATH','/kaggle/working/banklens_experiments_artifacts.zip')
